# Evidencia Evaluativa 1: obtener y preparar datos desde Hugging Face

**Integrantes:** Jaider Morales Bautista · Aleicer Vesga Rueda · Guillerno Leon Loaiza Mesa

En este notebook coordinamos las dos clases del proyecto:

1. `HFDatasetLoader` (`src/clases_po/hf_loader.py`): se conecta a Hugging Face y devuelve el dataset como DataFrame.
2. `Reportes` (`src/clases_po/reportes.py`): recibe un DataFrame y genera un resumen en `.txt`.

El flujo es: **cargar → transformar → resumir**. El notebook se ejecuta automáticamente en GitHub Actions (`.github/workflows/ea1.yml`), así que el reporte `.txt` siempre corresponde a la última ejecución.

**Dataset:** [`pointe77/credit-card-transaction`](https://huggingface.co/datasets/pointe77/credit-card-transaction), split `train`. Son transacciones de tarjeta de crédito simuladas, con datos del cliente, del comercio y una marca que dice si la transacción fue fraude.

## 1. Importar las clases

Buscamos la raíz del proyecto para poder importar las clases desde `src/clases_po`, sin importar desde qué carpeta se ejecute el notebook.

In [ ]:
import sys 
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

## 2. Cargar el dataset desde Hugging Face

Usamos `max_rows=100` para que esta primera actividad sea liviana: solo necesitamos ver la estructura de los datos y probar una transformación. En la actividad 2 se usa una muestra mucho más grande para entrenar los modelos.

In [ ]:
loader = HFDatasetLoader(
    dataset_name="pointe77/credit-card-transaction",
    split="train",
    max_rows=100,
    filename="datos/pointe77/credit-card-transaction.csv",
)

In [ ]:
df = loader.load()
df.head(2)

In [ ]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

## 3. Transformaciones aplicadas

Hicimos cuatro transformaciones sobre el DataFrame cargado:

1. **Renombrar columnas al español**, para que el reporte y el análisis sean más fáciles de leer para el grupo.
2. **Eliminar `Unnamed: 0`**, porque es solo un índice que quedó del archivo original y no aporta información.
3. **Rellenar los nulos de `codigo_postal_comerciante` con -1**. Así no perdemos esas filas y el valor -1 deja claro que el dato no existía.
4. **Calcular la `edad`** del cliente al momento de la transacción, restando el año de nacimiento al año de la transacción. Es un cálculo aproximado, porque no tiene en cuenta si la persona ya cumplió años ese año.

In [ ]:
# Diccionario con la traducción de las columnas a eEspañol
traduccion_columnas = {
    'trans_date_trans_time': 'fecha_hora_transaccion',
    'cc_num': 'num_tarjeta',
    'merchant': 'establecimiento',
    'category': 'categoria',
    'amt': 'monto',
    'first': 'nombre',
    'last': 'apellido',
    'gender': 'genero',
    'street': 'calle',
    'city': 'ciudad',
    'state': 'estado',
    'zip': 'codigo_postal',
    'lat': 'latitud',
    'long': 'longitud',
    'city_pop': 'poblacion_ciudad',
    'job': 'trabajo',
    'dob': 'fecha_nacimiento',
    'trans_num': 'num_transaccion',
    'unix_time': 'tiempo_unix',
    'merch_lat': 'latitud_comerciante',
    'merch_long': 'longitud_comerciante',
    'is_fraud': 'es_fraude',
    'merch_zipcode': 'codigo_postal_comerciante'
}

# Aplicar el cambio de nombres
df.rename(columns=traduccion_columnas, inplace=True)

In [ ]:
# Eliminar la columna innecesaria
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

In [ ]:
# Rellenar valores nulos con -1 para identificarlos fácilmente
df['codigo_postal_comerciante'] = df['codigo_postal_comerciante'].fillna(-1)
df.info()

In [ ]:
# 1. Extraer solo los primeros 4 caracteres del texto (el año) y convertirlos a número entero
año_trans = df['fecha_hora_transaccion'].str[:4].astype(int)
año_nacimiento = df['fecha_nacimiento'].str[:4].astype(int)

# 2. Calcular la edad simplemente restando los años
df['edad'] = año_trans - año_nacimiento

# Verificar el resultado
print(df[['fecha_hora_transaccion', 'fecha_nacimiento', 'edad']].head())

## 4. Generar el resumen en `.txt`

Pasamos el DataFrame **ya transformado** a la clase `Reportes`. El archivo queda en `reportes/pointe77/credit-card-transaction.txt` y por eso muestra las columnas en español y la nueva columna `edad`.

In [ ]:
from clases_po import Reportes

In [ ]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

In [ ]:
ruta = reportes.guardar_reporte(
df, "reportes/pointe77/credit-card-transaction.txt", 
)
print(f"Reporte guardado en: {ruta}")

## Conclusión

El flujo funciona de punta a punta: la primera clase trae los datos desde Hugging Face, el notebook los transforma y la segunda clase deja el resumen en un archivo de texto. Este mismo mecanismo de carga y reporte es la base que reutilizamos en la actividad 2.